# 01 — Bayesian Coin-Flip Foundation

**Purpose:** before calibrating a 5-parameter nonlinear car-following model, build
intuition for Bayesian inference on the simplest possible example: estimating the
bias of a coin from flip outcomes.

Concepts introduced here carry directly into the IDM calibration notebook:
- **Prior**: what we believe about a parameter before seeing data
- **Likelihood**: how probable the observed data is, given a parameter value
- **Posterior**: updated belief after combining prior + likelihood
- **MCMC (NUTS)**: how PyMC actually draws samples from the posterior when there's
  no closed-form solution (which will be the case for IDM)
- **Point estimate vs. distribution**: a classical MLE gives you one number; a
  Bayesian posterior gives you a whole distribution — including how *uncertain*
  that number is. This is exactly the gap that matters later when a classical
  least-squares fit of IDM's `s0` parameter looks confident but is actually
  unidentifiable in free-flow traffic.


In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)


## 1. Generate synthetic coin-flip data

We'll simulate a biased coin with true bias `p_true = 0.7` (probability of heads)
and flip it `n_flips` times.

In [1]:
p_true = 0.7
n_flips = 40

flips = rng.binomial(n=1, p=p_true, size=n_flips)
n_heads = flips.sum()
print(f"Observed {n_heads} heads out of {n_flips} flips (raw proportion = {n_heads/n_flips:.3f})")


NameError: name 'rng' is not defined

## 2. Build the Bayesian model

- **Prior**: `p ~ Beta(2, 2)` — mildly informative, centered at 0.5, allows the
  data to move it freely.
- **Likelihood**: `Binomial(n_flips, p)` observed = n_heads.

This has a closed-form posterior (Beta is conjugate to Binomial), but we'll sample
it with MCMC anyway since that's the general-purpose tool we'll need for IDM,
where no closed form exists.

In [ ]:
with pm.Model() as coin_model:
    p = pm.Beta("p", alpha=2, beta=2)
    obs = pm.Binomial("obs", n=n_flips, p=p, observed=n_heads)

    idata_coin = pm.sample(2000, tune=1000, chains=4, random_seed=RANDOM_SEED)

az.summary(idata_coin, var_names=["p"])


## 3. Inspect the posterior

Note the posterior is a *distribution*, not a single number. The 94% HDI
(highest density interval) tells us the range of plausible bias values given
the data — this is the uncertainty quantification a classical point estimate
throws away.

In [ ]:
az.plot_posterior(idata_coin, var_names=["p"], ref_val=p_true)
plt.title("Posterior over coin bias p (red line = true value)")
plt.show()


In [ ]:
az.plot_trace(idata_coin, var_names=["p"])
plt.tight_layout()
plt.show()


## 4. Effect of sample size on posterior width

Run the same inference with `n_flips = 5` vs `n_flips = 500` to see how the
posterior narrows as data accumulates. This is the same mechanism behind the
free-flow vs. congested identifiability gap in the IDM notebook: **not all data
is equally informative about all parameters** — here it's about *quantity*,
in the IDM case it's about *regime* (congested traffic exercises the `s0`
parameter; free flow barely does).

In [ ]:
def run_coin_model(n_flips, p_true=0.7, seed=0):
    rng_local = np.random.default_rng(seed)
    flips = rng_local.binomial(1, p_true, n_flips)
    n_heads = flips.sum()
    with pm.Model():
        p = pm.Beta("p", alpha=2, beta=2)
        pm.Binomial("obs", n=n_flips, p=p, observed=n_heads)
        idata = pm.sample(1000, tune=500, chains=2, progressbar=False, random_seed=seed)
    return idata

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharex=True)
for ax, n in zip(axes, [5, 40, 500]):
    idata_n = run_coin_model(n, seed=n)
    samples = idata_n.posterior["p"].values.flatten()
    ax.hist(samples, bins=30, density=True, alpha=0.7)
    ax.axvline(p_true, color="red", linestyle="--")
    ax.set_title(f"n_flips = {n}")
fig.suptitle("Posterior narrows as evidence accumulates")
fig.tight_layout()
plt.show()


## Takeaways -> carried into the IDM notebooks

1. A posterior distribution communicates confidence, not just a best guess.
2. MCMC (NUTS in PyMC) lets us sample posteriors even when there's no closed form.
3. How much a parameter's posterior narrows depends on how much the *data actually
   constrains it* — which motivates checking per-parameter identifiability
   explicitly in the IDM model, rather than trusting a single least-squares
   number.

Next: `02_linear_regression_bridge.ipynb` extends this to a *continuous*,
multi-parameter model (slope + intercept + noise) — the natural bridge from
a single proportion to the 5-parameter IDM calibration.
